# ⚽ How Reliable Are Football Betting Odds?
### Margins, calibration and a simple prediction — Python project, EM Normandie

| | |
|---|---|
| **Members** | Youssef Charrat · Khalil El Joumail · Ayoub Oukadi · El Mehdi Benchehba |
| **Dataset** | `closing_odds.csv` — closing odds and results of 479k football matches (2005–2015) |

**Who wrote what** *(every code cell also starts with an `# Authors:` line)*

| Section | Author(s) |
|---|---|
| 1. Introduction | All members |
| 2. Packages | All members |
| 3. Data cleaning and preparation | All members |
| 4. Exploratory data analysis | All members |
| 5. Feature engineering | All members |
| 6. Prediction | All members |
| 7. Summary | All members |

**Contents:** 1 Introduction · 2 Packages · 3 Data cleaning · 4 Exploratory analysis · 5 Feature engineering · 6 Prediction · 7 Summary · AI declaration · References

# 1. Introduction

### 1.1 Context
Before a football match, bookmakers publish **decimal odds** for a *home win*, a *draw* and an *away win*. The odds just before kick-off are the **closing odds**.

- An odd of `2.00` means a bet of 1 returns 2, and it **implies a probability** of `1 / 2.00 = 50%`.
- If we add the three implied probabilities of one match, we get **more than 100%**. The extra part is the **overround** (the *margin*): the bookmaker's built-in profit.

### 1.2 Business question
> **How reliable are bookmakers' odds, how much do they charge, and can we predict results better than the odds themselves?**

### 1.3 Sub-questions
1. How big is the bookmaker margin, and does it change with the **year** or the **number of bookmakers** (competition)?
2. Are the odds **calibrated**? (When the odds say 70%, does it really happen 70% of the time?)
3. Does a simple prediction rule, built on past matches, do better than the odds on future matches?

### 1.4 Why it matters
- **Bettors** want to know where they lose the least.
- **Bookmakers** must price their margin and risk correctly.
- **Regulators** care about how much of every stake goes back to customers.

### 1.5 The data
479,440 matches × 19 columns, from 2005-01-01 to 2015-06-30.

| Column | Meaning |
|---|---|
| `match_id`, `league`, `match_date` | identifier, "Country: Competition", date |
| `home_team`, `away_team`, `home_score`, `away_score` | teams and final score |
| `avg_odds_home_win`, `avg_odds_draw`, `avg_odds_away_win` | **average** closing odds over all bookmakers |
| `max_odds_*` and `top_bookie_*` | **best** odds available, and the bookmaker offering them |
| `n_odds_*` | number of bookmakers quoting the match |

# 2. Packages required
Same toolbox as in the lectures: `pandas` and `numpy` for the data, `matplotlib` and `seaborn` for the charts.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")     # nicer default style for all charts

# 3. Data cleaning and preparation

## 3.1 Load the data from a URL
We read the file with `pd.read_csv()` and a URL (as required).

⚠️ **The original file is compressed (gzip) even though it is called `.csv`**, so we add `compression="gzip"`.  
The `try / except` also works if the file was uploaded as a normal text file.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
DATA_URL = "https://raw.githubusercontent.com/charseph/football-odds-project/main/closing_odds.csv"    # raw link of our GitHub repository

def load_data(url):
    """Read the odds file from a URL (compressed file first, normal CSV as backup)."""
    try:
        return pd.read_csv(url, compression="gzip")
    except OSError:
        return pd.read_csv(url)

df_raw = load_data(DATA_URL)
print("Shape (rows, columns):", df_raw.shape)

## 3.2 First look at the data

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
display(df_raw.head(10))      # first 10 rows
display(df_raw.tail(10))      # last 10 rows

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
df_raw.info()                 # column names and data types

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
display(df_raw.describe().round(2))     # quick statistics of the numeric columns

**What we notice** in `describe()`:
- `avg_odds_home_win` and `avg_odds_away_win` have a **minimum of 0** — impossible, decimal odds must be above 1.
- The maximum score is over **100 goals** — clearly a data error.
- `match_date` is not a date type yet.

Let's check missing values and duplicates:

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
print("Missing values per column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print()
print("Duplicated rows:", df_raw.duplicated().sum())
print("Duplicated match_id:", df_raw["match_id"].duplicated().sum())

## 3.3 Checking the problems with filters
We use **Boolean filters** (as in Lecture 3) to count the suspicious rows *before* removing anything.
We use a **list comprehension** (Lecture 2) to get the list of odds columns.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
# list comprehension: all the "avg_odds..." and "max_odds..." columns
odds_cols = [c for c in df_raw.columns if c.startswith("avg_odds") or c.startswith("max_odds")]
print(odds_cols)

# 1) impossible odds (<= 1)
bad_odds = df_raw[(df_raw[odds_cols] <= 1).any(axis=1)]
print("\nRows with an odd <= 1:", len(bad_odds))

# 2) absurd scores (> 15 goals for one team)
big_scores = df_raw[(df_raw["home_score"] > 15) | (df_raw["away_score"] > 15)]
print("Rows with more than 15 goals for one team:", len(big_scores))
display(big_scores[["match_date", "home_team", "away_team", "home_score", "away_score"]].head())

# 3) impossible margin: the overround should be around 1.05 - 1.15
valid = df_raw[(df_raw[odds_cols] > 1).all(axis=1)]
overround_raw = 1 / valid["avg_odds_home_win"] + 1 / valid["avg_odds_draw"] + 1 / valid["avg_odds_away_win"]
print("Rows with overround < 1.0 or > 1.5:", ((overround_raw < 1.0) | (overround_raw > 1.5)).sum())

These rows are **very few** compared with 479,440 matches, so we can safely remove them.

## 3.4 Cleaning function
All the cleaning steps are in one function, so we can rerun them easily:

| Step | What | Why |
|---|---|---|
| 1 | `match_date` → real date | needed for dates |
| 2 | remove odds ≤ 1 | impossible |
| 3 | remove scores > 15 | data errors |
| 4 | remove overround outside [1.0, 1.5] | not a credible market |
| 5 | fill remaining missing bookmaker names with `"Unknown"` | keep the row |
| 6 | split `league` into `country` and `competition` | one variable per column |

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
def clean_data(df):
    df = df.copy()
    print("Rows at start:", len(df))

    # 1. convert the date
    df["match_date"] = pd.to_datetime(df["match_date"])

    # 2. remove impossible odds
    df = df[(df[odds_cols] > 1).all(axis=1)]
    print("After removing odds <= 1:", len(df))

    # 3. remove absurd scores
    df = df[(df["home_score"] <= 15) & (df["away_score"] <= 15)]
    print("After removing scores > 15:", len(df))

    # 4. remove impossible margins
    overround = 1 / df["avg_odds_home_win"] + 1 / df["avg_odds_draw"] + 1 / df["avg_odds_away_win"]
    df = df[(overround >= 1.0) & (overround <= 1.5)]
    print("After removing strange overround:", len(df))

    # 5. missing bookmaker names
    df = df.fillna(value={"top_bookie_home_win": "Unknown", "top_bookie_away_win": "Unknown"})

    # 6. split "Country: Competition"
    df["country"] = df["league"].str.split(": ").str[0]
    df["competition"] = df["league"].str.split(": ").str[1]

    # sort by date (oldest first) and renumber the rows
    return df.sort_values("match_date").reset_index(drop=True)

df = clean_data(df_raw)
print("\nMissing values left:", df.isnull().sum().sum())
print("Rows kept: {:.2%}".format(len(df) / len(df_raw)))

## 3.5 Basic new columns (including dates)
We create the columns that almost every analysis needs:

- **Dates (Lecture 6):** `year`, `month`, `weekday` from `match_date`.
- `result`: `H` home win, `D` draw, `A` away win.
- `overround` and the **margin-free probabilities** `prob_home`, `prob_draw`, `prob_away` (implied probabilities divided by the overround, so they add up to 100%).

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
def add_basic_columns(df):
    df = df.copy()

    # --- dates ---
    df["year"] = df["match_date"].dt.year
    df["month"] = df["match_date"].dt.month
    df["weekday_num"] = df["match_date"].dt.dayofweek          # 0 = Monday ... 6 = Sunday
    df["weekday"] = df["match_date"].dt.day_name()             # "Monday", ...

    # --- result of the match ---
    df["result"] = "A"                                                         # away win by default
    df.loc[df["home_score"] == df["away_score"], "result"] = "D"               # draw
    df.loc[df["home_score"] > df["away_score"], "result"] = "H"                # home win
    df["home_win"] = (df["result"] == "H").astype(int)                         # 1 if home win, else 0

    # --- odds -> probabilities ---
    inv_home = 1 / df["avg_odds_home_win"]
    inv_draw = 1 / df["avg_odds_draw"]
    inv_away = 1 / df["avg_odds_away_win"]
    df["overround"] = inv_home + inv_draw + inv_away
    df["margin_pct"] = (df["overround"] - 1) * 100
    df["prob_home"] = inv_home / df["overround"]
    df["prob_draw"] = inv_draw / df["overround"]
    df["prob_away"] = inv_away / df["overround"]
    return df

df = add_basic_columns(df)
display(df[["match_date", "weekday", "home_team", "away_team", "home_score", "away_score",
            "result", "overround", "prob_home", "prob_draw", "prob_away"]].head())

# 4. Exploratory data analysis
For each step: **what we do → chart → what we see**.

## 4.1 Summary statistics
We start with the numbers of Lecture 5: mean, median and standard deviation.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
print("Goals per team in a match:")
display(df[["home_score", "away_score"]].agg(["mean", "median", "std"]).round(2))

print("Share of each result:")
display(df["result"].value_counts(normalize=True).round(3))

print("Share of matches by year:")
display(df.groupby("year").size())

**What we see**
- A team scores on average **1.54 goals at home** and **1.20 away**. The median is 1 for both, and the standard deviation is similar (1.35 and 1.22).
- Home teams win **45.2%** of matches, away teams **29.8%**, and **25.0%** end in a draw.
- The number of matches grows a lot: from about **21,000 in 2005** to about **70,000 in 2014** (2015 only has six months). The biggest jump is in 2009 (29,796 → 42,889). So later years are much better covered than early ones.

## 4.2 Matches per year, and results vs expectations
**What we do:** count the matches per year, and compare the **real** share of home wins / draws / away wins with the share the **market expected** (average of the margin-free probabilities).

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# left: matches per year
per_year = df.groupby("year").size()
axes[0].bar(per_year.index, per_year.values, color="steelblue")
axes[0].set_title("Number of matches per year (2015 = only 6 months)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Number of matches")

# right: real results vs market expectation
real = df["result"].value_counts(normalize=True)[["H", "D", "A"]]
expected = df[["prob_home", "prob_draw", "prob_away"]].mean()
expected.index = ["H", "D", "A"]
x = np.arange(3)
axes[1].bar(x - 0.2, real.values, width=0.4, label="Real", color="steelblue")
axes[1].bar(x + 0.2, expected.values, width=0.4, label="Expected by the odds", color="lightgray", edgecolor="black")
axes[1].set_xticks(x)
axes[1].set_xticklabels(["Home win", "Draw", "Away win"])
axes[1].set_title("Real results vs what the odds expected")
axes[1].set_ylabel("Share of matches")
axes[1].legend()

plt.tight_layout()
plt.show()
print(pd.DataFrame({"real": real, "expected": expected}).round(3))

**What we see**
- *Left:* the dataset grows strongly until 2013–2014. Results for recent years are therefore more reliable than for 2005.
- *Right:* the odds expected **44.1% home wins** but **45.2%** really happened (+1.1 points). Draws were expected 25.7% vs 25.0% real, and away wins 30.1% vs 29.8% real.
- So the odds are **close to reality**, but slightly **under-estimate the home team** overall.

## 4.3 Home advantage: by competition and by weekday
**What we do:**
- *Left:* share of home wins in the 10 biggest competitions (`groupby`).
- *Right:* share of home wins by **year and weekday** as a **heatmap** (like the crime heatmap of Lecture 6).

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# left: home-win rate in the 10 biggest competitions
top_leagues = df["league"].value_counts().head(10).index
home_rate = df[df["league"].isin(top_leagues)].groupby("league")["home_win"].mean().sort_values()
axes[0].barh(home_rate.index, home_rate.values, color="teal")
axes[0].axvline(df["home_win"].mean(), color="red", linestyle="--", label="All matches")
axes[0].set_title("Share of home wins - 10 biggest competitions")
axes[0].set_xlabel("Share of home wins")
axes[0].set_ylabel("Competition")
axes[0].legend()

# right: heatmap year x weekday
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
table = df.pivot_table(index="year", columns="weekday", values="home_win", aggfunc="mean")[days]
sns.heatmap(table, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[1])
axes[1].set_title("Share of home wins by year and weekday")
axes[1].set_xlabel("Weekday")
axes[1].set_ylabel("Year")

plt.tight_layout()
plt.show()

**What we see**
- *Left:* home advantage depends on the competition. **Club friendlies have the fewest home wins (40.2%)**, then English League Two (42.1%), League One (43.1%) and the Championship (43.2%). The highest are international friendlies (47.3%), the Chilean Primera Division (46.9%) and the Dutch Eerste Divisie (46.3%). The red line is the overall average (45.2%).
- *Right:* the weekday hardly matters. Home-win shares stay between **0.42 and 0.48** for every weekday and year (overall: Tuesday 43.7% is the lowest, Sunday 45.8% the highest). The few higher or lower cells (e.g. Monday 2008: 0.41) come from small numbers of matches.
- Most matches are played at the weekend: **Saturday and Sunday = about 64%** of all matches.

## 4.4 The bookmaker margin
**What we do:** look at the margin (`overround − 1`, in %) with a histogram, over the years, and against the number of bookmakers quoting the match. Then a **boxplot** by country (Lecture 4).

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
print("Average margin: {:.2f}%   |   median: {:.2f}%".format(df["margin_pct"].mean(), df["margin_pct"].median()))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) histogram of the margin
axes[0].hist(df["margin_pct"], bins=60, color="indianred")
axes[0].set_title("Distribution of the bookmaker margin")
axes[0].set_xlabel("Margin (%)")
axes[0].set_ylabel("Number of matches")

# 2) margin by year
margin_year = df[df["year"] < 2015].groupby("year")["margin_pct"].mean()
axes[1].plot(margin_year.index, margin_year.values, marker="o", color="darkred")
axes[1].set_title("Average margin by year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Average margin (%)")

# 3) margin by number of bookmakers
margin_n = df.groupby("n_odds_home_win")["margin_pct"].agg(["mean", "size"])
margin_n = margin_n[margin_n["size"] >= 500]          # ignore very small groups
axes[2].plot(margin_n.index, margin_n["mean"], marker="o", color="darkblue")
axes[2].set_title("Margin vs number of bookmakers")
axes[2].set_xlabel("Number of bookmakers quoting the match")
axes[2].set_ylabel("Average margin (%)")

plt.tight_layout()
plt.show()

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
# boxplot of the margin in the 8 biggest countries
top_countries = df["country"].value_counts().head(8).index
plt.figure(figsize=(12, 5))
sns.boxplot(data=df[df["country"].isin(top_countries)], x="country", y="margin_pct",
            order=df[df["country"].isin(top_countries)].groupby("country")["margin_pct"].median().sort_values().index,
            showfliers=False, color="lightsteelblue")
plt.title("Bookmaker margin by country (8 biggest)")
plt.xlabel("Country")
plt.ylabel("Margin (%)")
plt.show()

**What we see**
- *Histogram:* the average margin is **10.6%** (median 11.0%): out of every 100 staked, bookmakers are built to keep about 10.6.
- *By year:* the margin **fell from 11.9% (2005) to 10.3% (2014)**, so betting became about 1.5 points cheaper for customers.
- *By number of bookmakers:* the more bookmakers quote a match, the lower the margin: **12.3% with 1 bookmaker** and **8.0% with 29 bookmakers** (strong negative link). This is a correlation, not proof: big matches attract more bookmakers *and* better prices.
- *Boxplot:* among the 8 biggest countries the median margin goes from **10.2% (European competitions)** and 10.3% (France) up to **11.3% (Italy, Spain)**, with England at 10.6%. The differences between big countries are small compared with the effect of the number of bookmakers.

## 4.5 Are the odds calibrated?
**What we do:** we cut the matches into 10 equal groups according to the market's probability (`pd.qcut`, Lecture 6). In each group we compare the **average probability from the odds** with the **share that really happened**.
If the odds are perfect, the points are on the dashed diagonal.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
def calibration_table(data, prob_col, outcome):
    groups = pd.qcut(data[prob_col], 10)                       # 10 groups with the same number of matches
    happened = (data["result"] == outcome).astype(int)         # 1 if the outcome happened
    table = pd.DataFrame({"from_odds": data[prob_col], "real": happened}).groupby(groups, observed=True).mean()
    return table

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, col, outcome, title in zip(axes, ["prob_home", "prob_draw", "prob_away"], ["H", "D", "A"],
                                   ["Home win", "Draw", "Away win"]):
    t = calibration_table(df, col, outcome)
    ax.plot(t["from_odds"], t["real"], marker="o", label="Real")
    top = t["from_odds"].max()
    ax.plot([0, top], [0, top], "k--", label="Perfect odds")
    ax.set_title("Calibration - " + title)
    ax.set_xlabel("Probability given by the odds")
    ax.set_ylabel("Share that really happened")
    ax.legend()
plt.tight_layout()
plt.show()

display(calibration_table(df, "prob_home", "H").round(3))

**What we see**
- The points are **very close to the diagonal**, so the odds are good probability estimates.
- But there is a clear pattern for **home wins**: the strongest favourites win **more** than the odds say (top group: **72.6% from the odds vs 77.2% real**), and the weakest home teams win **less** (16.3% vs 15.3%).
- The same happens for **away wins** (top group 61.5% vs 64.8%; bottom group 9.7% vs 7.7%) and for **draws** (when a draw looks unlikely, 15.9% from the odds vs 12.9% real).
- This is the well-known **favourite–longshot bias**: the market gives a bit too much chance to outsiders and too little to strong favourites.

## 4.6 Outliers (Lecture 5 method)
In Lecture 5 an outlier is a value far from the centre, for example **more than 3 standard deviations above the mean**.
**What we do:** find the matches with extreme away odds (big outsiders) and check whether they are errors or real.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
mean_odds = df["avg_odds_away_win"].mean()
std_odds = df["avg_odds_away_win"].std()
limit = mean_odds + 3 * std_odds
outliers = df[df["avg_odds_away_win"] > limit]

print("Mean = {:.2f}, std = {:.2f}  ->  outlier limit = {:.1f}".format(mean_odds, std_odds, limit))
print("Number of outliers:", len(outliers), "({:.2%} of matches)".format(len(outliers) / len(df)))
print("Away win expected by the odds in these matches: {:.1%}".format(outliers["prob_away"].mean()))
print("Away win really happened in these matches:      {:.1%}".format((outliers["result"] == "A").mean()))
display(outliers[["match_date", "league", "home_team", "away_team", "avg_odds_away_win", "result"]].head())

**What we see**
- **8,510 matches (1.8%)** have away odds above 12.3 (more than 3 standard deviations above the mean of 3.97). They are big outsiders, not errors: for example Genoa – Venezia with odds of 15.
- In these matches the odds expected an away win **5.7%** of the time, but it really happened only **2.7%**. The extreme outsiders are therefore **over-priced** (same favourite–longshot bias as before).
- **Decision:** we **keep** these outliers. They are real matches, and removing them would hide exactly the effect we study. (We only removed impossible values in Section 3.)

## 4.7 Main findings of the exploration

1. **Data:** 479,016 matches after cleaning (99.9% kept), much better covered after 2008.
2. **Results:** home 45.2%, draw 25.0%, away 29.8%. The odds slightly **under-estimate home wins** (44.1% expected). Home advantage varies by competition (40% in friendlies up to 47%) but **not by weekday**.
3. **Margin:** about **10.6%**, falling from 11.9% to 10.3% over the decade, and **much lower when many bookmakers compete** (12.3% → 8.0%).
4. **Calibration:** the odds are accurate but show a **favourite–longshot bias**: strong favourites win more than implied (77% vs 73%), big outsiders win less.
5. **Outliers:** extreme outsiders (1.8% of matches) are real and clearly over-priced, so we keep them.

# 5. Feature engineering
We create **11 new features** from existing columns, with tools from the course (new columns, binning with `cut` / `qcut`, `merge`).

| # | New feature | How it is built | Why |
|---|---|---|---|
| 1 | `total_goals` | home_score + away_score | goals in the match |
| 2 | `goal_diff` | home_score − away_score | size of the victory |
| 3 | `favourite` | outcome with the highest probability (H, D or A) | who the market expects |
| 4 | `fav_prob` | probability of the favourite | how sure the market is |
| 5 | `upset` | 1 if the favourite did not win | surprise results |
| 6 | `prob_gap` | \|prob_home − prob_away\| | how one-sided the match is |
| 7 | `n_bookmakers` | mean of the three `n_odds_*` columns | size of the market |
| 8 | `is_weekend` | 1 if Saturday or Sunday | calendar effect |
| 9 | `fav_strength` | `pd.cut` on `fav_prob`: Weak / Medium / Strong | **binning** (Lecture 6) |
| 10 | `margin_level` | `pd.qcut` on the margin: Low / Medium / High | **binning with equal groups** |
| 11 | `league_home_rate` | past home-win rate of the league, **merged** from the training period | **merge** (Lecture 5) |

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
def add_features(df):
    df = df.copy()
    df["total_goals"] = df["home_score"] + df["away_score"]
    df["goal_diff"] = df["home_score"] - df["away_score"]

    prob_cols = ["prob_home", "prob_draw", "prob_away"]
    df["fav_prob"] = df[prob_cols].max(axis=1)
    df["favourite"] = df[prob_cols].idxmax(axis=1).map({"prob_home": "H", "prob_draw": "D", "prob_away": "A"})
    df["upset"] = (df["favourite"] != df["result"]).astype(int)
    df["prob_gap"] = (df["prob_home"] - df["prob_away"]).abs()

    df["n_bookmakers"] = df[["n_odds_home_win", "n_odds_draw", "n_odds_away_win"]].mean(axis=1)
    df["is_weekend"] = (df["weekday_num"] >= 5).astype(int)

    # binning (Lecture 6)
    df["fav_strength"] = pd.cut(df["fav_prob"], bins=[0, 0.45, 0.60, 1.0], labels=["Weak", "Medium", "Strong"])
    df["margin_level"] = pd.qcut(df["margin_pct"], 3, labels=["Low", "Medium", "High"])
    return df

df = add_features(df)
display(df[["total_goals", "goal_diff", "favourite", "fav_prob", "upset", "prob_gap",
            "n_bookmakers", "is_weekend", "fav_strength", "margin_level"]].head())

### Merge: past home-win rate of each league
We compute the home-win rate of every league **only on matches before 2014** (the training period), then **merge** it back to all matches with a **left join** (we keep every match, Lecture 5).
Using only the past avoids "cheating" in the prediction part. Leagues that do not exist before 2014 get the overall average.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
train_period = df[df["match_date"] < "2014-01-01"]

# one row per league: home-win rate in the training period
league_stats = train_period.groupby("league")["home_win"].mean().reset_index()
league_stats = league_stats.rename(columns={"home_win": "league_home_rate"})
print("Leagues in the table:", len(league_stats))

rows_before = len(df)
df = df.merge(league_stats, on="league", how="left")           # left join: keep all matches
print("Rows before / after the merge:", rows_before, "/", len(df))

print("Matches without a league rate (new leagues):", df["league_home_rate"].isnull().sum())
df["league_home_rate"] = df["league_home_rate"].fillna(train_period["home_win"].mean())
df[["league", "league_home_rate"]].drop_duplicates().head()

### Quick check: do the new features make sense?
**What we do:** upset rate by `fav_strength`, and average margin by `margin_level`.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

upset_rate = df.groupby("fav_strength", observed=True)["upset"].mean() * 100
axes[0].bar(upset_rate.index.astype(str), upset_rate.values, color="indianred")
axes[0].set_title("How often does the favourite NOT win?")
axes[0].set_xlabel("Strength of the favourite (from the odds)")
axes[0].set_ylabel("Favourite did not win (%)")

by_margin = df.groupby("margin_level", observed=True)["margin_pct"].mean()
axes[1].bar(by_margin.index.astype(str), by_margin.values, color="slateblue")
axes[1].set_title("Average margin in each margin level")
axes[1].set_xlabel("Margin level")
axes[1].set_ylabel("Average margin (%)")

plt.tight_layout()
plt.show()
print(upset_rate.round(1))

**What we see**
- *Left:* the favourite does **not** win in **59.2%** of the matches where it is *weak*, **46.8%** when *medium* and only **26.8%** when *strong*. So `fav_strength` really measures how uncertain a match is.
- *Right:* the three margin levels have an average margin of **8.9% (Low), 10.9% (Medium) and 12.1% (High)**, which is what we expect from equal-sized groups.
- The merge kept all **479,016 rows**. About 8,900 matches belong to leagues that did not exist before 2014; they received the overall average home-win rate.

# 6. Prediction: can we beat the odds?

**Goal.** Predict the result (`H`, `D`, `A`) of a match *before* it is played.

**Train / test split by time** (like predicting the future from the past):
- **Training:** matches before 2014 → we *learn* from them
- **Test:** matches from 2014 on → we *evaluate* on them

**Two measures:**
- **Accuracy** = share of correct predictions.
- **Log loss** = quality of the *probabilities* (**lower is better**). A perfect-confidence model has 0; guessing 1/3 each gives 1.099.

**Three simple rules** (only `pandas`, no machine-learning package):
1. Always predict **home win**.
2. Predict the **favourite** of the odds.
3. **Learn from the past:** in the training period, check how often an outcome *really* happened for each probability range (a table made with `pd.cut` + `groupby`), and use these real rates as new probabilities.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
train = df[df["match_date"] < "2014-01-01"]
test = df[df["match_date"] >= "2014-01-01"].copy()
print("Training matches:", len(train))
print("Test matches:    ", len(test))

# --- Rule 1: always home win ---
acc_home = (test["result"] == "H").mean()

# --- Rule 2: the favourite of the odds ---
acc_fav = (test["favourite"] == test["result"]).mean()

# --- Log loss (by hand) ---
def log_loss_simple(result, p_home, p_draw, p_away):
    """Average of -log(probability given to what really happened). Lower = better."""
    p_real = np.where(result == "H", p_home, np.where(result == "D", p_draw, p_away))
    return -np.mean(np.log(p_real))

ll_market = log_loss_simple(test["result"], test["prob_home"], test["prob_draw"], test["prob_away"])
print("\nAlways home win -> accuracy: {:.4f}".format(acc_home))
print("Favourite       -> accuracy: {:.4f} | log loss: {:.4f}".format(acc_fav, ll_market))

### Rule 3: learn the real rates from the past
For each outcome, we cut the probability into ranges (0–10%, 10–20%, …) and compute, **in the training period only**, how often the outcome really happened. We then apply this table to the test matches and make the three probabilities add up to 100%.

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
edges = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]       # probability ranges

def real_rates(train_data, prob_col, outcome):
    """Real frequency of an outcome in each probability range (training data only)."""
    ranges = pd.cut(train_data[prob_col], edges, labels=False)
    return (train_data["result"] == outcome).groupby(ranges).mean()

def apply_rates(data, prob_col, rates):
    """Replace each probability by the real rate of its range."""
    ranges = pd.cut(data[prob_col], edges, labels=False)
    return ranges.map(rates).fillna(data[prob_col])             # if a range is missing, keep the odds

# learn on the training period
rates_home = real_rates(train, "prob_home", "H")
rates_draw = real_rates(train, "prob_draw", "D")
rates_away = real_rates(train, "prob_away", "A")

# apply on the test period
new = pd.DataFrame({"H": apply_rates(test, "prob_home", rates_home),
                    "D": apply_rates(test, "prob_draw", rates_draw),
                    "A": apply_rates(test, "prob_away", rates_away)})
new = new.div(new.sum(axis=1), axis=0)                          # make each row add up to 100%

test["pred_new"] = new.idxmax(axis=1)                           # most likely outcome
acc_new = (test["pred_new"] == test["result"]).mean()
ll_new = log_loss_simple(test["result"], new["H"], new["D"], new["A"])
print("Rule 3 -> accuracy: {:.4f} | log loss: {:.4f}".format(acc_new, ll_new))

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba
results = pd.DataFrame({
    "Rule": ["1. Always home win", "2. Favourite of the odds (= the market)", "3. Real rates learned from the past"],
    "Accuracy": [acc_home, acc_fav, acc_new],
    "Log loss": [np.nan, ll_market, ll_new],
}).set_index("Rule").round(4)
display(results)

# Which results does rule 3 predict? (share of each real result that was predicted as H / D / A)
print("Rule 3: real result (rows) vs prediction (columns)")
display(pd.crosstab(test["result"], test["pred_new"], normalize="index").round(2))

**What we see**
- **Rule 1 (always home win):** right 44.6% of the time. This is the minimum we must beat.
- **Rule 2 (the favourite of the odds):** right **52.6%** of the time, with a log loss of **0.9786**. This is our benchmark: **the market itself**.
- **Rule 3 (real rates learned from the past):** accuracy 52.3% and log loss **0.9809** → slightly **worse** than the odds.
- Rule 3 **never predicts a draw** (column `D` = 0): a draw is almost never the single most likely result.

**Why does rule 3 not win?** Two reasons:
1. The pattern learned before 2014 **did not fully hold afterwards**. Home teams won 45.4% of matches in the training period, but only **44.6%** in the test period. The market had adjusted: home advantage was falling.
2. Our ranges are wide (10 points), so we lose precision.

**Conclusion:** with simple rules we **cannot beat the odds**. Closing odds already contain almost all available information, and the biases we found (favourite–longshot) are too small and too unstable to exploit easily.

### Bonus (optional, goes beyond the course): logistic regression
The project says we may learn tools not seen in class. Here is the same test with a classic machine-learning model from `scikit-learn`. **If you do not want to explain it at the defense, delete this section.**

In [ ]:
# Authors: Youssef Charrat, Khalil El Joumail, Ayoub Oukadi, El Mehdi Benchehba   (external help: scikit-learn documentation)
from sklearn.linear_model import LogisticRegression

features = ["prob_home", "prob_draw", "prob_away", "overround", "league_home_rate"]
model = LogisticRegression(max_iter=1000)
model.fit(train[features], train["result"])                    # learn on the training period

proba = model.predict_proba(test[features])                    # columns are in the order: A, D, H
pred = model.predict(test[features])

ll_model = log_loss_simple(test["result"], proba[:, 2], proba[:, 1], proba[:, 0])
print("Logistic regression -> accuracy: {:.4f} | log loss: {:.4f}".format((pred == test["result"]).mean(), ll_model))
print("Market (odds)       -> accuracy: {:.4f} | log loss: {:.4f}".format(acc_fav, ll_market))

**What we see**
- Logistic regression reaches **52.5% accuracy** and a log loss of **0.9793**; the market has **52.6% and 0.9786**.
- The model is **practically equal to the odds, but not better**. This confirms the previous result: the odds are very hard to beat.
- Adding extra columns (`overround`, `league_home_rate`) did not help, because the odds already contain this information.

# 7. Summary

### 7.1 Problem and approach
**Question:** how reliable are football betting odds, how much do bookmakers charge, and can we predict results better than the odds?

**What we did:**
1. Loaded 479,440 matches from a URL (compressed file) and **cleaned** them (52 impossible odds, 18 absurd scores and 354 impossible margins removed → 479,016 rows kept).
2. **Explored** results, home advantage, margin, calibration and outliers with `groupby`, boxplots, heatmaps and binning.
3. Built **11 new features** (including binning and a merge).
4. **Predicted** results with a time-based split (train before 2014, test 2014 to June 2015) and compared with the odds.

### 7.2 Insights
1. **Margin:** about **10.6%**, falling from **11.9% (2005) to 10.3% (2014)**. It is **lowest when many bookmakers compete**: 12.3% with one bookmaker, 8.0% with 29.
2. **Calibration:** odds are accurate but show a **favourite–longshot bias**. Strong favourites win more than implied (77.2% vs 72.6%), big outsiders win less (extreme outsiders: 2.7% real vs 5.7% expected).
3. **Home advantage** is 45.2% overall and depends on the competition (40% in friendlies, 47% in internationals), but **not on the weekday**. It **declined** in 2014–2015 (44.6%).
4. **Prediction:** the favourite rule is right **52.6%** of the time (vs 44.6% for "always home"). Neither our learned-rates rule (log loss 0.9809) nor logistic regression (0.9793) **beat the odds** (0.9786).

### 7.3 Managerial implications
- **For bettors:** the margin is the main cost (about 10% of stakes). Prefer **popular matches with many bookmakers** (about 8% margin), and avoid bets on big outsiders, which are the worst value.
- **For bookmakers:** margins are highest where few bookmakers quote a match, so these markets are the most profitable but also the easiest for competitors to attack. Prices should correct the favourite–longshot bias and the changing home advantage.
- **For regulators / consumers:** the cost of betting varies by about 4 points depending on competition; clearer information on the payout rate (1 / overround) would help.
- **For analysts:** the odds are the benchmark. A model that only equals them is the normal result in an efficient market.

### 7.4 Limitations
- Only **closing odds** (no odds movement), only 2005–2015, and coverage grows over time (uneven between leagues).
- Club friendlies are included, where "home advantage" is not meaningful.
- We removed the margin by dividing by the overround; other methods could change the size of the bias.
- One single train/test split, and simple prediction rules (the 10-point ranges are coarse).
- No information on injuries, line-ups or team quality beyond the odds.
- The link between number of bookmakers and margin is a correlation, not proven causation.

### 7.5 Ideas to go further
Add team form (using `shift()` on previous matches), study one league only, use `resample` to follow the margin by quarter, or test more models.

### 7.6 Course tools used (for the defense)
| Lecture | What we used |
|---|---|
| L1–L2 Python | functions, list comprehension, dictionaries (`map`), `try / except` |
| L3 Pandas | `read_csv`, `head / tail / describe / info`, Boolean filters, `isnull`, `fillna`, `groupby`, `sort_values`, new columns |
| L4 Visualization | `matplotlib` subplots, bar / barh / hist / line, `seaborn` boxplot and heatmap |
| L5 EDA | mean / median / std, outliers (mean + 3 std), `merge` (left join) |
| L6 Dates | `to_datetime`, `.dt` (year, month, weekday), `pd.cut`, `pd.qcut`, heatmap by weekday and year |

# Declaration of Generative AI and AI-assisted technologies in the writing process

During the preparation of this work the author(s) used **Claude (Anthropic)** in order to `[DESCRIBE HONESTLY HOW YOU USED IT, e.g. understand the project instructions, find the right pandas functions, structure the notebook, improve the readability and language of the text]`. After using this tool/service, the author(s) reviewed and edited the content as needed and take(s) full responsibility for the content of the publication.

# References
- Dataset: `closing_odds.csv` — closing odds and results of football matches, 2005–2015. Hosted at https://raw.githubusercontent.com/charseph/football-odds-project/main/closing_odds.csv
- Course lectures L01–L06 (Pr. S. Bouraga, EM Normandie).
- pandas documentation — https://pandas.pydata.org/docs/
- seaborn and matplotlib documentation — https://seaborn.pydata.org, https://matplotlib.org
- scikit-learn documentation (bonus section only) — https://scikit-learn.org/stable/
- Claude (Anthropic), AI assistant used as described in the declaration above — https://claude.ai